# Data Structures for Machine Learning: Engineering & Algorithms Lab

Machine learning systems do not run on pure mathematics alone; they depend fundamentally on data structures designed for spatial indexing, constant-time retrieval, cache locality, and directed acyclic graph (DAG) scheduling.

### Table of Contents
1. [Memory Hierarchy & Cache Locality: Contiguous Arrays vs. Linked Pointers](#1-memory-hierarchy--cache-locality-contiguous-arrays-vs-linked-pointers)
2. [Hash Maps & Feature Embedding Lookup Tables](#2-hash-maps--feature-embedding-lookup-tables)
3. [Circular Ring Buffers for Streaming Replay & Sliding Windows](#3-circular-ring-buffers-for-streaming-replay--sliding-windows)
4. [KD-Trees for Fast Spatial k-Nearest Neighbors (k-NN)](#4-kd-trees-for-fast-spatial-k-nearest-neighbors-k-nn)
5. [Min-Heaps for Streaming Top-K Retrieval in Recommendations](#5-min-heaps-for-streaming-top-k-retrieval-in-recommendations)
6. [Directed Acyclic Graphs (DAGs) & Topological Scheduling in Neural Networks](#6-directed-acyclic-graphs-dags--topological-scheduling-in-neural-networks)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Loaded Data Structures for ML lab environment.")

## 1. Memory Hierarchy & Cache Locality: Contiguous Arrays vs. Linked Pointers
A contiguous memory block (NumPy array) allows CPU L1/L2 caches to prefetch cache lines (64 bytes), resulting in dramatic performance speedups over fragmented pointer-chasing structures.

In [ ]:
# Benchmark sequential traversal: Contiguous array vs. Python list of pointers
N = 2_000_000
np_arr = np.ones(N, dtype=np.float64)
py_list = list(np_arr)

# Measure NumPy contiguous array traversal
t0 = time.perf_counter()
s1 = np.sum(np_arr)
t_numpy = time.perf_counter() - t0

# Measure Python list pointer traversal
t1 = time.perf_counter()
s2 = sum(py_list)
t_list = time.perf_counter() - t1

print(f"Contiguous Array Time: {t_numpy * 1000:.2f} ms")
print(f"Fragmented List Time:  {t_list * 1000:.2f} ms")
print(f"Cache Locality Speedup: {t_list / t_numpy:.1f}x")

## 2. Hash Maps & Feature Embedding Lookup Tables
In recommendation systems and NLP, categorical features (User ID, Token ID) map to dense latent vectors via $\mathcal{O}(1)$ average-time hash lookups.

In [ ]:
# Embedding table mapping categorical vocabulary tokens to dense vectors
embedding_dim = 8
vocab = ["transformer", "attention", "gradient", "tensor", "loss"]

# Hash Map embedding layer
embedding_table = {
    token: np.random.randn(embedding_dim).astype(np.float32)
    for token in vocab
}

query_token = "attention"
vector = embedding_table.get(query_token)
print(f"Embedding for '{query_token}' (dim {embedding_dim}):")
print(np.round(vector, 3))

## 3. Circular Ring Buffers for Streaming Replay & Sliding Windows
Circular queues allow rolling buffers to operate with strictly bounded $\mathcal{O}(1)$ insertion and memory footprint.

In [ ]:
class RingBuffer:
    def __init__(self, capacity):
        self.cap = capacity
        self.buf = [None] * capacity
        self.head = 0
        self.tail = 0
        self.size = 0

    def push(self, val):
        self.buf[self.tail] = val
        self.tail = (self.tail + 1) % self.cap
        if self.size < self.cap:
            self.size += 1
        else:
            self.head = (self.head + 1) % self.cap

    def get_window(self):
        return [self.buf[(self.head + i) % self.cap] for i in range(self.size)]

rb = RingBuffer(capacity=5)
for item in [10, 20, 30, 40, 50, 60, 70]:
    rb.push(item)

print("Rolling Window Buffer (Capacity 5 after 7 inserts):")
print(rb.get_window())

## 4. KD-Trees for Fast Spatial k-Nearest Neighbors (k-NN)
Brute force nearest neighbor search scales as $\mathcal{O}(N)$ distance calculations. Spatial partitioning with a KD-Tree reduces expected query time to $\mathcal{O}(\log N)$ by recursively splitting points along median hyperplanes.

In [ ]:
from sklearn.neighbors import KDTree

# Generate 2D spatial points
np.random.seed(42)
points = np.random.uniform(0, 100, size=(1000, 2))
kdtree = KDTree(points, leaf_size=10)

query = np.array([[50.0, 50.0]])
dist, indices = kdtree.query(query, k=5)

plt.figure(figsize=(7, 6))
plt.scatter(points[:, 0], points[:, 1], c="lightgray", s=15, label="All Data Points")
plt.scatter(points[indices[0], 0], points[indices[0], 1], c="#1f77b4", s=60, label="5 Nearest Neighbors")
plt.scatter(query[0, 0], query[0, 1], c="red", s=100, marker="X", label="Query Point [50, 50]")
plt.title("KD-Tree Nearest Neighbor Spatial Query", fontweight="bold")
plt.xlabel("Dimension X1")
plt.ylabel("Dimension X2")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Min-Heaps for Streaming Top-K Retrieval in Recommendations
Finding the top-$K$ scoring items among $N$ candidates using full sorting takes $\mathcal{O}(N \log N)$. Maintaining a min-heap of size $K$ does this in $\mathcal{O}(N \log K)$ time.

In [ ]:
import heapq

# Stream of 1,000,000 recommendation candidate scores
N = 1_000_000
K = 5
scores = np.random.uniform(0, 1000, size=N)

t0 = time.perf_counter()
# Min-heap maintains smallest of top-K at root
top_k_heap = []
for s in scores:
    if len(top_k_heap) < K:
        heapq.heappush(top_k_heap, s)
    elif s > top_k_heap[0]:
        heapq.heapreplace(top_k_heap, s)
t_heap = time.perf_counter() - t0

t1 = time.perf_counter()
top_k_sort = np.sort(scores)[-K:][::-1]
t_sort = time.perf_counter() - t1

print(f"Heap Time O(N log K):   {t_heap * 1000:.2f} ms")
print(f"Full Sort Time O(N log N): {t_sort * 1000:.2f} ms")
print(f"Top {K} Scores: {np.round(sorted(top_k_heap, reverse=True), 2)}")

## 6. Directed Acyclic Graphs (DAGs) & Topological Scheduling in Neural Networks
Deep learning computational frameworks (PyTorch FX, ONNX, TensorFlow Graph) model models as DAGs. Evaluating operations in valid forward and backward dependency order requires topological sorting (Kahn's Algorithm).

In [ ]:
# Simple Kahn's algorithm for DAG topological sort
def topological_sort(nodes, edges):
    in_degree = {n: 0 for n in nodes}
    adj = {n: [] for n in nodes}
    for src, dst in edges:
        adj[src].append(dst)
        in_degree[dst] += 1

    queue = [n for n in nodes if in_degree[n] == 0]
    order = []
    while queue:
        curr = queue.pop(0)
        order.append(curr)
        for neighbor in adj[curr]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    return order

operators = ["Input_Tokens", "Embedding", "Self_Attention", "LayerNorm", "MLP", "Logits"]
dependencies = [
    ("Input_Tokens", "Embedding"),
    ("Embedding", "Self_Attention"),
    ("Self_Attention", "LayerNorm"),
    ("LayerNorm", "MLP"),
    ("MLP", "Logits")
]

execution_plan = topological_sort(operators, dependencies)
print("Deterministic Operator Execution Schedule:")
for step, op in enumerate(execution_plan, 1):
    print(f"  Step {step}: {op}")